In [ ]:
# Se cargan mensajes financieros en texto bruto para preparar una representación que un modelo pueda procesar.

import pandas as pd

dataframe = pd.read_csv("../data/sentences.csv.zip", compression="zip")
dataframe[["phrase", "target"]].head()

In [ ]:
# ¿Qué diferencias superficiales se pueden eliminar sin cambiar el contenido de cada mensaje?

dataframe["normalized_text"] = dataframe["phrase"].str.lower().str.split().str.join(" ")
dataframe[["phrase", "normalized_text"]].head()

In [ ]:
# Se separa cada documento en tokens para que el texto se convierta en unidades procesables.

from nltk.tokenize import word_tokenize

dataframe["tokens"] = dataframe["normalized_text"].map(word_tokenize)
dataframe[["normalized_text", "tokens"]].head()

In [ ]:
# Se conservan tokens alfabéticos y se hace visible la información que la regla descarta.

dataframe["tokens"] = dataframe["tokens"].map(
    lambda tokens: [token for token in tokens if token.isalpha()]
)
dataframe["processed_text"] = dataframe["tokens"].map(" ".join)
dataframe[["normalized_text", "processed_text"]].head()

In [ ]:
# ¿Cómo se transforma la lista de tokens en una matriz documento-término apta para clasificación?

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(min_df=2)
document_term_matrix = vectorizer.fit_transform(dataframe["processed_text"])
vocabulary = pd.DataFrame(
    {"token": vectorizer.get_feature_names_out(), "document_frequency": (document_term_matrix > 0).sum(axis=0).A1}
)
document_term_matrix.shape, vocabulary.head()

In [ ]:
# Se guardan documentos procesados, vocabulario y matriz para que otro taller pueda entrenar un clasificador.

import json
from pathlib import Path
from scipy import sparse

submission_dir = Path("../submission")
dataframe.drop(columns="tokens").to_csv(submission_dir / "tokenized_sentences.csv", index=False)
vocabulary.to_csv(submission_dir / "vocabulary.csv", index=False)
sparse.save_npz(submission_dir / "document_term_matrix.npz", document_term_matrix)
with (submission_dir / "matrix_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "documents": document_term_matrix.shape[0],
            "terms": document_term_matrix.shape[1],
            "minimum_document_frequency": 2,
        },
        file,
        indent=2,
    )